# ISLES 2015 SISS clean-room reproduction

This notebook runs entirely in Google Colab. Nothing needs to be installed on your computer. It downloads the historical dataset, verifies it, and runs the reconstructed MLP/SVM pipeline.

**Important:** the paper does not state how its one FLAIR slice per patient was chosen. The `paper-like` experiment below uses the maximum-ground-truth-lesion slice as an explicit **oracle** reconstruction assumption. It is not a leakage-free clinical protocol.


In [ ]:
!rm -rf HybridStrokeSeg-Reproduction
!git clone -q https://github.com/NTinkicht/HybridStrokeSeg-Reproduction.git
%cd HybridStrokeSeg-Reproduction
!python -m pip install -q -e '.[notebook]'


## 1. Download and validate the organizer re-archive


In [ ]:
!python scripts/download_isles2015.py --extract
!python scripts/validate_dataset.py --geometry


## 2. Quick end-to-end smoke run
This uses 2,000 pixels per class so we can verify the whole pipeline before the slower paper-size run.


In [ ]:
!python scripts/run_reproduction.py --protocol paper-like --target-per-class 2000 --models mlp svm --mlp-solver adam --mlp-max-iter 150 --output-dir outputs/smoke


In [ ]:
import pandas as pd
display(pd.read_csv('outputs/smoke/summary_metrics.csv'))
display(pd.read_csv('outputs/smoke/per_case_metrics.csv'))


## 3. Paper-size clean-room reconstruction
This requests the manuscript's approximately 15,000 lesion and 15,000 non-lesion training pixels and runs both models. The RBF-SVM can take noticeably longer than the smoke test. The MLP architecture is 3x100 sigmoid, but LBFGS is used because scikit-learn does not implement the paper's scaled conjugate-gradient optimizer.


In [ ]:
!python scripts/run_reproduction.py --protocol paper-like --target-per-class 15000 --models mlp svm --output-dir outputs/paper_like


In [ ]:
display(pd.read_csv('outputs/paper_like/summary_metrics.csv'))
display(pd.read_csv('outputs/paper_like/per_case_metrics.csv'))


## 4. Leakage-free slice-selection sensitivity check
This selects the middle non-empty brain slice without consulting the lesion mask. It can miss lesions; that contrast is useful because the original slice-selection rule is unknown.


In [ ]:
!python scripts/run_reproduction.py --protocol leakage-free-slice --target-per-class 15000 --models mlp svm --output-dir outputs/leakage_free_slice
display(pd.read_csv('outputs/leakage_free_slice/summary_metrics.csv'))


## Reproducibility guardrail
Do not describe a close Dice score as proof that the original source code has been recovered. Read `docs/reconstruction_assumptions.md`: weighted mean, directional descriptor, preprocessing, slice selection, SVM hyperparameters, morphology and the MLP optimizer contain explicit reconstruction assumptions that require sensitivity testing.
